In [0]:
%run ../0-common/env-config

In [0]:
%run ../0-common/bronze_helpers

In [0]:
source_file = f"{landing_folfer_path}/results/"
table_name = f"{catalog_name}.{bronze_schema}.results"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, DateType

results_schema = StructType([
    StructField("date", DateType()),
    StructField("raceName", StringType()),
    StructField("round", IntegerType()),
    StructField("season", IntegerType()),
    StructField("url", StringType()),
    StructField("constructorId", StringType()),
    StructField("driverId", StringType()),
    StructField("grid", IntegerType()),
    StructField("laps", IntegerType()),
    StructField("number", IntegerType()),
    StructField("points", FloatType()),
    StructField("position", IntegerType()),
    StructField("positionText", StringType()),
    StructField("status", StringType())
])

In [0]:
results_df = (
    spark.read.format('json')
    .option('inferSchema', True)
    .schema(results_schema)
    .load(source_file)
)

In [0]:
results_df_final = add_ingestion_metadata(results_df)

In [0]:
(
    results_df_final.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
%sql
select season, count(*)
from formula1.bronze.results
group by season
order by season